In [0]:
import requests

dbutils.widgets.text("api_url","https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_day.geojson")
api_url=dbutils.widgets.get("api_url")
response=requests.get(api_url)

In [0]:
dbutils.widgets.text('catalog_name','youtube_dev','youtube_dev')
catalog_name=dbutils.widgets.get('catalog_name')
print(catalog_name)


In [0]:
%python
import requests
import json
from pyspark.sql import Row
import datetime

spark.sql(f"use catalog {catalog_name}")
spark.sql("use schema bronze")
spark.sql("create volume if not exists earthquake_data")

host="https://earthquake.usgs.gov"
base_path="/earthquakes/feed/v1.0"
feed="summary/all_day.geojson"
url=f"{host}{base_path}/{feed}"
response=requests.get(url)
if(response.status_code!=200):
    raise Exception(f"Error in getting data from {url}")
earthquake_data= response.json()
current_date=datetime.datetime.now().strftime('%Y-%m-%d')
dbutils.fs.put(f"/Volumes/{catalog_name}/bronze/earthquake_data/earthquake_data{current_date}.json",json.dumps(earthquake_data),overwrite=True)
# Extract and flatten features
rows = []
for feature in earthquake_data['features']:
    props = feature['properties']
    coords = feature['geometry']['coordinates']
    rows.append(Row(
        id=feature['id'],
        magnitude=float(props.get('mag')) if props.get('mag') is not None else None,
        place=props.get('place'),
        time=int(props.get('time')) if props.get('time') is not None else None,
        updated=int(props.get('updated')) if props.get('updated') is not None else None,
        url=props.get('url'),
        longitude=float(coords[0]),
        latitude=float(coords[1]),
        depth=float(coords[2])
    ))

df = spark.createDataFrame(rows)
display(df)

In [0]:
%sql


DROP TABLE IF EXISTS youtube_dev.silver.__materialization_mat_584f73a7_96f7_4447_bea6_d7e50501c4f7_earthquake_data_1;

